# Did the Prevention Collaborative Work? Interrupted Time Series and Difference-in-Differences

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/nhsn-dialysis-analytics/blob/main/notebooks/03_its_did_evaluation.ipynb)

**Portfolio project 3 of 5** | Role focus: Data Analyst I, NHSN Dialysis Event prevention and surveillance support

---
## How this notebook relates to notebooks 01 and 02
| Notebook | Role of the collaborative |
|---|---|
| 01 data quality | Built the world, including a simulated CVC-reduction collaborative that starts in January 2024. Never analyzed it |
| 02 rate models | Used "collaborative participation" only as a **nuisance covariate** so rates were not biased. Its coefficient (about 0.72) was a by-product, not an evaluation |
| **03 this notebook** | Makes the collaborative **the question**: did it reduce events, by how much, and how sure are we? |

Same seed and same code as notebook 01, so the same facilities, events, and planted errors. Only **validated** records are used
for the main analysis, and a section near the end shows what happens if you skip validation.

## The known answer (used for evaluation only)
In the simulator, 40% of facilities were assigned at random to the collaborative. From January 2024 their **CVC rates are
multiplied by 0.70**; AVF, AVG, and "Other" access rates are unchanged. So the true rate ratio is **0.70 for CVC events**, **1.00 for
non-CVC events (a placebo outcome)**, and a diluted value for all events combined. No method sees these numbers while fitting.

## Data statement
**No PHI and no real NHSN data.** Everything is simulated and rates are illustrative. The collaborative is fictional.

## What this notebook does
| Step | Method | Question it answers |
|---|---|---|
| 3 | Descriptive monthly rates | What does the picture look like before any model? |
| 4 | **Interrupted time series** (segmented Poisson regression, autocorrelation-robust SEs) | What changed at the start date in the treated facilities, and did the untreated ones change too? |
| 5 | **Difference-in-differences**: 2x2, two-way fixed effects, and an **event study** | How much did the treated change *relative to* comparison facilities? Were trends parallel before? |
| 6 | Placebo outcome, placebo date, **permutation test**, cluster bootstrap | Could this result be chance or a modeling artifact? |
| 7 | **Selection bias demonstration** (regression to the mean) | What if facilities join *because* their rates were high? |
| 8 | Heterogeneity, sensitivity to data quality, events averted | Who benefited, does validation matter, what is the public health impact? |

## 0. Setup

In [ ]:
import sys, subprocess, importlib, warnings, os
warnings.filterwarnings("ignore")

def _ensure(pip_name, import_name=None):
    """Install a package only if it is missing (Colab already has most of these)."""
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for pip_name in ["numpy", "pandas", "scipy", "matplotlib", "statsmodels", "openpyxl"]:
    _ensure(pip_name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf
from IPython.display import display

SEED = 42
rng = np.random.default_rng(SEED)        # used ONLY by the notebook 01 recap, so the data match exactly
np.random.seed(SEED)
rng2 = np.random.default_rng(SEED + 1)   # separate stream for permutations and bootstraps below

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})
OUT = "outputs_03"
os.makedirs(OUT, exist_ok=True)
print("pandas", pd.__version__, "| statsmodels", sm.__version__)

## 1. Rebuild the notebook 01 world
Compact copy of notebook 01 (configuration, generator, error injection, validation rules), collapsed in Colab.
**Do not change the seed or the order.** In a real repository these functions would live in `src/`.

In [ ]:
#@title Recap from notebook 01: config { display-mode: "form" }
N_FACILITIES = 160
MONTHS = pd.period_range("2023-01", "2024-12", freq="M")      # 24 reporting months
ACCESS = ["AVF", "AVG", "CVC_tunneled", "CVC_nontunneled", "Other"]
EVENT_TYPES = ["positive_blood_culture", "iv_antimicrobial_start", "pus_redness_swelling"]

# Illustrative baseline rates per 100 patient-months, by access type (columns follow EVENT_TYPES order).
BASE_RATE = {
    "AVF":             [0.15, 0.45, 0.10],
    "AVG":             [0.35, 0.90, 0.25],
    "CVC_tunneled":    [1.10, 2.00, 0.40],
    "CVC_nontunneled": [2.50, 3.50, 0.80],
    "Other":           [0.30, 0.80, 0.20],
}

LATE_DAYS = 30           # submission later than this many days after month end counts as late
DUP_WINDOW_DAYS = 21     # same patient + same event type within this window = duplicate (illustrative rule)
MIN_PLAUSIBLE_YEAR = 2015

# Probability each record is hit by an injected error (facility-month level, then event level).
ERR_RATES = dict(
    missing=0.020, patient_typo=0.015, sum_mismatch=0.020, zero_access=0.030, late=0.030, inflation=0.025,
    duplicate=0.020, date_outside=0.015, date_implausible=0.007, access_swapped=0.015,
)
print(f"{N_FACILITIES} facilities x {len(MONTHS)} months = {N_FACILITIES*len(MONTHS):,} facility-months")

In [ ]:
#@title Recap from notebook 01: generator { display-mode: "form" }
def make_facilities(n, rng):
    states = ["FL", "GA", "TX", "CA", "NY", "OH", "PA", "IL", "NC", "MI"]
    state_network = {s: f"NW-{(i % 6) + 1:02d}" for i, s in enumerate(states)}   # synthetic network codes
    fac = pd.DataFrame({"facility_id": [f"FAC{i+1:04d}" for i in range(n)]})
    fac["state"] = rng.choice(states, n)
    fac["network"] = fac["state"].map(state_network)
    fac["ownership"] = rng.choice(["Large_chain", "Nonprofit", "Independent"], n, p=[0.60, 0.25, 0.15])
    fac["urban_rural"] = rng.choice(["Urban", "Rural"], n, p=[0.78, 0.22])
    size = rng.lognormal(np.log(65), 0.45, n) * np.where(fac["urban_rural"] == "Rural", 0.7, 1.0)
    fac["mean_census"] = np.clip(size, 15, 250).round().astype(int)
    base_mix = np.array([0.62, 0.13, 0.17, 0.02, 0.06])
    mix = np.vstack([rng.dirichlet(base_mix * 60) for _ in range(n)])
    for j, a in enumerate(ACCESS):
        fac[f"mix_{a}"] = mix[:, j]
    fac["log_effect"] = rng.normal(0, 0.35, n)             # latent facility risk (ground truth, never used as a feature)
    fac["in_collaborative"] = rng.random(n) < 0.40
    return fac


def make_clean_denominator(fac, months, rng):
    rows, mix_cols = [], [f"mix_{a}" for a in ACCESS]
    for f in fac.itertuples(index=False):
        mix = np.array([getattr(f, c) for c in mix_cols]); mix = mix / mix.sum()
        drift = rng.normal(0.002, 0.003)                    # slow census growth/decline
        for t, m in enumerate(months):
            start = m.to_timestamp()
            end = (m + 1).to_timestamp() - pd.Timedelta(days=1)
            total = max(8, int(round(f.mean_census * (1 + drift * t) * (1 + rng.normal(0, 0.025)))))
            counts = rng.multinomial(total, mix)
            delay = int(np.clip(rng.gamma(2.0, 5.0), 3, 28))   # typical on-time submission
            rows.append({"facility_id": f.facility_id, "month": start,
                         "report_date": end + pd.Timedelta(days=delay), "patients_total": total,
                         **{f"n_{a}": int(c) for a, c in zip(ACCESS, counts)}})
    return pd.DataFrame(rows)


def make_clean_events(fac, den, rng):
    info = fac.set_index("facility_id")[["log_effect", "in_collaborative", "mean_census"]].to_dict("index")
    collab_start, rows = pd.Timestamp("2024-01-01"), []
    for r in den.itertuples(index=False):
        f = info[r.facility_id]
        ndays = r.month.days_in_month
        post = bool(f["in_collaborative"]) and r.month >= collab_start
        shock = rng.gamma(20, 1 / 20)                         # monthly extra-Poisson variation
        pool = int(f["mean_census"] * 1.6)
        for a in ACCESS:
            n_a = getattr(r, f"n_{a}")
            if n_a == 0:
                continue
            mult = 0.7 if (post and a.startswith("CVC")) else 1.0
            for k, et in enumerate(EVENT_TYPES):
                lam = n_a * BASE_RATE[a][k] / 100 * np.exp(f["log_effect"]) * shock * mult
                for _ in range(rng.poisson(lam)):
                    rows.append({"facility_id": r.facility_id, "month": r.month,
                                 "patient_id": f"{r.facility_id}-P{rng.integers(1, pool + 1):04d}",
                                 "event_type": et, "access_type": a,
                                 "event_date": r.month + pd.Timedelta(days=int(rng.integers(0, ndays)))})
    return pd.DataFrame(rows)


def enforce_dup_window(ev, window=DUP_WINDOW_DAYS):
    """Clean data should already obey the duplicate-event rule: drop a repeat for the same patient/type inside the window."""
    ev = ev.sort_values(["facility_id", "patient_id", "event_type", "event_date"]).reset_index(drop=True)
    keep, last_key, last_date = np.ones(len(ev), bool), None, None
    for i, (fid, pid, et, d) in enumerate(zip(ev.facility_id, ev.patient_id, ev.event_type, ev.event_date)):
        key = (fid, pid, et)
        if key == last_key and (d - last_date).days <= window:
            keep[i] = False
            continue
        last_key, last_date = key, d
    return ev[keep].reset_index(drop=True)


fac = make_facilities(N_FACILITIES, rng)
den_clean = make_clean_denominator(fac, MONTHS, rng)
ev_clean = enforce_dup_window(make_clean_events(fac, den_clean, rng))
ev_clean = ev_clean.sort_values(["facility_id", "month", "event_date"]).reset_index(drop=True)
ev_clean.insert(0, "event_id", [f"EV{i+1:06d}" for i in range(len(ev_clean))])

In [ ]:
#@title Recap from notebook 01: error injection { display-mode: "form" }
def _append_label(ev, mask, label):
    cur = ev.loc[mask, "err_event"]
    ev.loc[mask, "err_event"] = np.where(cur == "", label, cur + "|" + label)


def inject_errors(den_c, ev_c, rng, rates=ERR_RATES):
    den, ev = den_c.copy(), ev_c.copy()
    n = len(den)
    cnt_cols = [f"n_{a}" for a in ACCESS]
    month_end = den["month"] + pd.offsets.MonthEnd(0)

    T = pd.DataFrame({"facility_id": den.facility_id, "month": den.month})
    for k in ["missing", "patient_typo", "sum_mismatch", "zero_access", "late", "inflation"]:
        T[k] = rng.random(n) < rates[k]

    # --- facility-month errors on the denominator ---
    m = T.patient_typo.to_numpy()
    den.loc[m, ["patients_total"] + cnt_cols] = den.loc[m, ["patients_total"] + cnt_cols] * 10

    m = T.sum_mismatch.to_numpy()
    factor = 1 + rng.choice([-1, 1], m.sum()) * rng.uniform(0.08, 0.20, m.sum())
    den.loc[m, "patients_total"] = (den.loc[m, "patients_total"] * factor).round().astype(int)

    ev_by_key = ev_c.groupby(["facility_id", "month"])["access_type"].apply(list).to_dict()
    za_map = {}
    for i in np.flatnonzero(T.zero_access.to_numpy()):
        key = (den.at[i, "facility_id"], den.at[i, "month"])
        cands = sorted(a for a in set(ev_by_key.get(key, [])) if a != "AVF" and den_c.at[i, f"n_{a}"] > 0)   # sorted: set order varies between Python processes
        if not cands:
            T.at[i, "zero_access"] = False
            continue
        a = cands[int(rng.integers(len(cands)))]
        den.at[i, "n_AVF"] += den.at[i, f"n_{a}"]       # patients moved to AVF; total stays consistent
        den.at[i, f"n_{a}"] = 0
        za_map[key] = a

    m = T.late.to_numpy()
    den.loc[m, "report_date"] = month_end[m] + pd.to_timedelta(rng.integers(35, 91, m.sum()), unit="D")

    # --- event-level errors ---
    ev = ev.merge(den[["facility_id", "month", "report_date"]], on=["facility_id", "month"], how="left")
    ev["err_event"] = ""
    N = len(ev)
    k_out, k_imp, k_swap = (int(N * rates[x]) for x in ("date_outside", "date_implausible", "access_swapped"))
    perm = rng.permutation(N)
    i_out, i_imp = perm[:k_out], perm[k_out:k_out + k_imp]
    i_swap = perm[k_out + k_imp:k_out + k_imp + k_swap]
    i_dup = rng.choice(perm[k_out + k_imp + k_swap:], int(N * rates["duplicate"]), replace=False)

    shift = (rng.choice([-1, 1], k_out) * rng.integers(35, 70, k_out))
    ev.loc[i_out, "event_date"] = ev.loc[i_out, "event_date"].to_numpy() + pd.to_timedelta(shift, unit="D").to_numpy()
    half = len(i_imp) // 2
    ev.loc[i_imp[:half], "event_date"] = pd.Timestamp("1900-01-01")
    ev.loc[i_imp[half:], "event_date"] = ev.loc[i_imp[half:], "report_date"] + pd.Timedelta(days=180)
    ev.loc[i_swap, "access_type"] = [rng.choice([x for x in ACCESS if x != a]) for a in ev.loc[i_swap, "access_type"]]
    for idx, lab in [(i_out, "date_outside_month"), (i_imp, "date_implausible"), (i_swap, "access_swapped")]:
        ev.loc[idx, "err_event"] = lab

    dups = ev.loc[i_dup].copy()
    dups["event_id"] = "EVD" + dups["event_id"].str[2:]
    shifted = dups["event_date"] + pd.to_timedelta(rng.integers(1, 15, len(dups)), unit="D")
    dups["event_date"] = pd.concat([shifted, dups["month"] + pd.offsets.MonthEnd(0)], axis=1).min(axis=1)
    dups["err_event"] = "duplicate"

    # over-reporting: valid-looking extra events for new patient IDs, so no event-level rule can catch them
    extra, c = [], 0
    for i in np.flatnonzero(T.inflation.to_numpy()):
        r = den_c.iloc[i]
        w = np.array([r[f"n_{a}"] * sum(BASE_RATE[a]) for a in ACCESS], float)
        for _ in range(int(rng.integers(3, 8))):
            c += 1
            a = ACCESS[int(rng.choice(len(ACCESS), p=w / w.sum()))]
            extra.append({"event_id": f"EVX{c:06d}", "facility_id": r.facility_id, "month": r.month,
                          "patient_id": f"{r.facility_id}-X{c:05d}", "event_type": str(rng.choice(EVENT_TYPES)),
                          "access_type": a,
                          "event_date": r.month + pd.Timedelta(days=int(rng.integers(0, r.month.days_in_month))),
                          "report_date": den.at[i, "report_date"], "err_event": "inflated_valid"})
    ev = pd.concat([ev, dups, pd.DataFrame(extra)], ignore_index=True)

    # label events caused by facility-month problems, then apply missing denominators
    zero_flag = np.array([za_map.get((f, mo)) == a for f, mo, a in zip(ev.facility_id, ev.month, ev.access_type)])
    _append_label(ev, zero_flag, "den_zero_access")
    miss_keys = set(zip(T.loc[T.missing, "facility_id"], T.loc[T.missing, "month"]))
    _append_label(ev, np.array([k in miss_keys for k in zip(ev.facility_id, ev.month)]), "no_denominator")

    den_sub = den.loc[~T.missing.to_numpy()].reset_index(drop=True)
    truth_ev = ev[["event_id", "err_event"]].copy()
    ev_sub = ev.drop(columns="err_event").sort_values(["facility_id", "month", "event_id"]).reset_index(drop=True)
    T["has_fm_error"] = T[["patient_typo", "sum_mismatch", "zero_access", "late", "inflation"]].any(axis=1)
    return den_sub, ev_sub, truth_ev, T


den, ev, truth_ev, truth_fm = inject_errors(den_clean, ev_clean, rng)

In [ ]:
#@title Recap from notebook 01: rule registry { display-mode: "form" }
RULES = pd.DataFrame([
    ("R1_date_outside_month", "event", 2, "Event date falls outside the reporting month it was entered under",
     "Hard stop at data entry when event date is outside the selected reporting month",
     "State that the event date, not the entry date, determines the reporting month."),
    ("R2_date_implausible", "event", 3, f"Event date before {MIN_PLAUSIBLE_YEAR} or after the submission date",
     "Block future dates and dates before facility enrollment",
     "Remind users that placeholder dates (for example 01/01/1900) must never be used."),
    ("R3_no_denominator", "event", 3, "Event reported for a facility-month with no submitted denominator",
     "Prevent saving events for a month until its denominator is submitted (or show a blocking alert)",
     "Add a step: complete the monthly denominator before entering events."),
    ("R4_zero_access_denominator", "event", 2, "Event access type has zero patients in that month's denominator",
     "Cross-field warning when event access type has a zero denominator count",
     "Clarify that denominator access-type counts must reflect each patient's access on the first two working days."),
    ("R5_duplicate_21d", "event", 2, f"Same patient and event type repeated within {DUP_WINDOW_DAYS} days",
     "Possible-duplicate alert that shows the earlier event at entry",
     "Add a worked example of the repeat-event window to the form instructions (confirm against current protocol)."),
    ("R6_missing_month", "facility-month", 2, "Enrolled facility has no denominator for a reporting month",
     "Automated missing-data report and reminder to facilities",
     "Add a monthly 'no events to report' attestation pathway."),
    ("R7_events_exceed_patients", "facility-month", 3, "Events for an access type exceed patients in that access category",
     "Numerator/denominator consistency check on submit",
     "Explain how event access type maps to denominator categories."),
    ("R8a_access_sum_mismatch", "facility-month", 2, "Total patients does not equal the sum of access-type counts",
     "Auto-calculate the total from access-type fields (read-only total)",
     "Clarify that the total is the sum of the access-type rows."),
    ("R8b_patient_count_outlier", "facility-month", 2, "Patient count is an outlier vs the facility's own history (robust |z| > 5)",
     "Soft warning when census differs sharply from prior months, with confirm button",
     "Add guidance for confirming unusually large or small census values."),
    ("R9_late_submission", "facility-month", 1, f"Submission more than {LATE_DAYS} days after month end",
     "Timeliness dashboard and automatic reminders at day 20 and day 28",
     "State the submission deadline prominently on the form."),
], columns=["rule_id", "level", "severity", "description", "proposed_modification", "form_instruction_edit"])

In [ ]:
#@title Recap from notebook 01: validation functions { display-mode: "form" }
ACC_N = [f"n_{a}" for a in ACCESS]
EVENT_RULES = ["R1_date_outside_month", "R2_date_implausible", "R3_no_denominator", "R4_zero_access_denominator", "R5_duplicate_21d"]


def validate_events(ev, den):
    """Return the event line list with one boolean column per event-level rule."""
    e = ev.merge(den[["facility_id", "month"] + ACC_N], on=["facility_id", "month"], how="left", indicator=True)
    e["R3_no_denominator"] = e["_merge"].eq("left_only")
    a_idx = e["access_type"].map({a: i for i, a in enumerate(ACCESS)}).to_numpy()
    n_for_access = e[ACC_N].to_numpy(float)[np.arange(len(e)), a_idx]
    e["R4_zero_access_denominator"] = (~e["R3_no_denominator"]) & (np.nan_to_num(n_for_access, nan=1) == 0)
    e["R1_date_outside_month"] = e["event_date"].dt.to_period("M") != e["month"].dt.to_period("M")
    e["R2_date_implausible"] = (e["event_date"].dt.year < MIN_PLAUSIBLE_YEAR) | (e["event_date"] > e["report_date"])
    s = e.sort_values(["facility_id", "patient_id", "event_type", "event_date", "event_id"])
    same = (s[["facility_id", "patient_id", "event_type"]].shift().eq(s[["facility_id", "patient_id", "event_type"]]).all(axis=1))
    close = (s["event_date"] - s["event_date"].shift()).dt.days.between(0, DUP_WINDOW_DAYS)
    dup_ids = set(s.loc[same & close, "event_id"])
    e["R5_duplicate_21d"] = e["event_id"].isin(dup_ids)
    e["any_event_rule"] = e[EVENT_RULES].any(axis=1)
    return e.drop(columns=ACC_N + ["_merge"])


def validate_facility_months(den, ev_flags, months, fac_ids):
    grid = pd.MultiIndex.from_product([fac_ids, [m.to_timestamp() for m in months]],
                                      names=["facility_id", "month"]).to_frame(index=False)
    d = grid.merge(den, on=["facility_id", "month"], how="left")
    d["R6_missing_month"] = d["patients_total"].isna()
    present = ~d["R6_missing_month"]

    cnt = ev_flags.pivot_table(index=["facility_id", "month"], columns="access_type", values="event_id",
                               aggfunc="count", fill_value=0).reindex(columns=ACCESS, fill_value=0)
    cnt.columns = [f"ev_{a}" for a in ACCESS]
    d = d.merge(cnt.reset_index(), on=["facility_id", "month"], how="left")
    ev_cols = [f"ev_{a}" for a in ACCESS]
    d[ev_cols] = d[ev_cols].fillna(0).astype(int)
    d["events_total"] = d[ev_cols].sum(axis=1)

    d["R4_any"] = d.merge(ev_flags.groupby(["facility_id", "month"])["R4_zero_access_denominator"].any().rename("x").reset_index(),
                          on=["facility_id", "month"], how="left")["x"].fillna(False).astype(bool).to_numpy()
    d["R7_events_exceed_patients"] = present & (d[ev_cols].to_numpy() > d[ACC_N].fillna(0).to_numpy()).any(axis=1)
    d["R8a_access_sum_mismatch"] = present & (d["patients_total"] != d[ACC_N].sum(axis=1))
    lp = np.log(d["patients_total"].clip(lower=1))
    dev = lp - lp.groupby(d["facility_id"]).transform("median")
    mad = dev.abs().groupby(d["facility_id"]).transform("median")
    d["log_pat_dev"] = dev
    d["robust_z_patients"] = dev / np.maximum(1.4826 * mad, 0.03)       # floor avoids dividing by ~0
    d["R8b_patient_count_outlier"] = present & (d["robust_z_patients"].abs() > 5)
    d["delay_days"] = (d["report_date"] - (d["month"] + pd.offsets.MonthEnd(0))).dt.days
    d["R9_late_submission"] = present & (d["delay_days"] > LATE_DAYS)
    d["rule_fm_count"] = d[["R4_any", "R7_events_exceed_patients", "R8a_access_sum_mismatch",
                            "R8b_patient_count_outlier", "R9_late_submission"]].sum(axis=1)
    d["rule_fm_flag"] = d["rule_fm_count"] > 0
    return d


E = validate_events(ev, den)
D = validate_facility_months(den, E, MONTHS, fac.facility_id.tolist())

In [ ]:
print(f"Recap done: {len(fac)} facilities, {len(den):,} denominator rows, {len(ev):,} submitted events; "
      f"{int(E.any_event_rule.sum())} events and {int(D.rule_fm_flag.sum())} facility-months flagged by the rules")

## 2. Analysis panels
One row per facility-month. Outcomes:
- **CVC events** per CVC patient-month (the targeted outcome)
- **Non-CVC events** per non-CVC patient-month (placebo: the collaborative should not change these)
- **All events** per patient-month (the policy-relevant total, diluted by the unaffected access types)

Three versions of the same panel are built so Section 8 can show the cost of skipping validation: **validated** (main),
**as-submitted** (no validation), and **clean** (the unknowable truth, available only in simulation).

In [ ]:
CVC = ["CVC_tunneled", "CVC_nontunneled"]
INTERVENTION = pd.Timestamp("2024-01-01")
T0 = 12                                                    # month index of the first post-intervention month (Jan 2024)
FAC_ATTR = fac[["facility_id", "in_collaborative", "mean_census", "ownership", "urban_rural"]].rename(columns={"in_collaborative": "treated"})


def make_panel(den_df, ev_df):
    d = den_df[["facility_id", "month"] + ACC_N].copy()
    d["pm_cvc"] = d["n_CVC_tunneled"] + d["n_CVC_nontunneled"]
    d["pm_non"] = d[["n_AVF", "n_AVG", "n_Other"]].sum(axis=1)
    d["pm_all"] = d["pm_cvc"] + d["pm_non"]
    e = ev_df[["facility_id", "month", "access_type"]].copy()
    e["cvc"] = e["access_type"].isin(CVC).astype(int)
    ec = e.groupby(["facility_id", "month"]).agg(ev_cvc=("cvc", "sum"), ev_all=("cvc", "size")).reset_index()
    d = d.merge(ec, on=["facility_id", "month"], how="left").fillna({"ev_cvc": 0, "ev_all": 0})
    d["ev_non"] = d["ev_all"] - d["ev_cvc"]
    d = d.merge(FAC_ATTR, on="facility_id")
    d["t"] = (d["month"].dt.year - 2023) * 12 + d["month"].dt.month - 1
    d["post"] = (d["t"] >= T0).astype(int)
    d["treated"] = d["treated"].astype(int)
    for c in ["ev_cvc", "ev_all", "ev_non"]:
        d[c] = d[c].astype(int)
    return d

# validated records exactly as in notebook 02
bad_fm = D.R6_missing_month | D.rule_fm_flag
bad_keys = set(zip(D.loc[bad_fm, "facility_id"], D.loc[bad_fm, "month"]))
ev_ok = E[~E.any_event_rule & ~pd.Series(list(zip(E.facility_id, E.month))).isin(bad_keys).to_numpy()].copy()
d_ok = D[~bad_fm].copy()

pan = make_panel(d_ok, ev_ok)             # validated (main analysis)
pan_raw = make_panel(den, ev)             # as submitted, no validation
pan_clean = make_panel(den_clean, ev_clean)   # truth, simulation only

n_t = int(fac.in_collaborative.sum())
print(f"Treated (collaborative) facilities: {n_t} | comparison facilities: {len(fac) - n_t} | intervention: {INTERVENTION:%B %Y} (month index {T0})")
print(f"Validated panel: {len(pan):,} facility-months | CVC events {pan.ev_cvc.sum():,} | non-CVC events {pan.ev_non.sum():,}")
display(pan.head())

## 3. Look before modeling
Monthly rate per 100 patient-months, by group. In the treated group the CVC rate should drop after the start date; the comparison
group should not.

In [ ]:
def monthly(p, y, expo, treated):
    g = p[p.treated == treated].groupby("t").agg(ev=(y, "sum"), pm=(expo, "sum")).reset_index()
    g["rate"] = g.ev / g.pm * 100
    return g

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
for ax, (y, expo, title) in zip(axes, [("ev_cvc", "pm_cvc", "CVC events (targeted outcome)"), ("ev_non", "pm_non", "Non-CVC events (placebo outcome)")]):
    for tr, lab, col in [(1, "Collaborative facilities", "crimson"), (0, "Comparison facilities", "steelblue")]:
        g = monthly(pan, y, expo, tr)
        ax.plot(g.t, g.rate, marker="o", ms=3.5, color=col, label=lab)
    ax.axvline(T0 - 0.5, color="k", ls="--", lw=1)
    ax.set_title(title); ax.set_xlabel("Month index (0 = Jan 2023)"); ax.set_ylabel("Events per 100 patient-months")
axes[0].legend(frameon=False)
plt.tight_layout(); plt.show()

pre_post = (pan.groupby(["treated", "post"]).apply(lambda g: pd.Series({"cvc_rate_per_100pm": g.ev_cvc.sum() / g.pm_cvc.sum() * 100,
                                                                          "non_cvc_rate_per_100pm": g.ev_non.sum() / g.pm_non.sum() * 100}))
            .round(2))
pre_post.index = pd.MultiIndex.from_tuples([(("Collaborative" if a else "Comparison"), ("Post" if b else "Pre")) for a, b in pre_post.index])
display(pre_post)

## 4. Interrupted time series (ITS)
**Idea:** use only the collaborative facilities, summed to one monthly series, and ask whether the level or slope broke at the
start date. Segmented Poisson regression with a `log(patient-months)` offset:

```
log E[events_t] = b0 + b1*time + b2*post + b3*(time since start)*post + log(patient-months_t)
```
- `exp(b2)` = **level-change rate ratio** at the start month (the immediate effect)
- `exp(b3)` = **slope-change ratio per month** (a gradual effect)

**Standard errors:** monthly series can be serially correlated, which makes ordinary standard errors too small. The usual remedy is
Newey-West (HAC) standard errors, but with only 24 time points HAC is known to be unreliable, so we (1) test for autocorrelation
and overdispersion in the residuals first, (2) report model-based, quasi-Poisson, and HAC intervals side by side, and (3) **check
by simulation how often each interval actually contains the truth**. The choice of standard-error method is made from
these diagnostics, never from the estimated effect.

**Model choice is stated in advance:** the campaign's expected mechanism is an immediate step, so the "step only" model is the
pre-specified primary analysis; "step + slope" is a flexibility check. (Choosing among them after seeing the data would inflate false positives.)

**Control series check:** the same model on the comparison facilities should show *no* break. If it does, something else
changed in January 2024 (a coding change, a seasonal effect) and a single-group ITS would be misleading.

In [ ]:
def its_design(t, full=True, seasonal=False):
    t = np.asarray(t, float)
    post = (t >= T0).astype(float)
    X = pd.DataFrame({"const": 1.0, "time": t, "post": post})
    if full:
        X["time_since"] = (t - T0) * post
    if seasonal:
        X["sin12"] = np.sin(2 * np.pi * (t % 12) / 12); X["cos12"] = np.cos(2 * np.pi * (t % 12) / 12)
    return X

def series_of(p, y, expo, treated):
    return p[p.treated == treated].groupby("t").agg(ev=(y, "sum"), pm=(expo, "sum")).reset_index()

def fit_its(s, full=False, seasonal=False, cov="HAC"):
    m = sm.GLM(s.ev, its_design(s.t, full, seasonal), family=sm.families.Poisson(), offset=np.log(s.pm))
    if cov == "HAC":
        return m.fit(cov_type="HAC", cov_kwds={"maxlags": 3})
    if cov == "quasi":
        return m.fit(scale="X2")
    return m.fit()

def its_row(label, res):
    ci = res.conf_int()
    row = {"model": label, "level-change RR (exp b2)": np.exp(res.params["post"]), "95% low": np.exp(ci.loc["post", 0]),
           "95% high": np.exp(ci.loc["post", 1]), "AIC (unscaled Poisson)": np.nan}
    if "time_since" in res.params.index:
        row["slope-change RR per month"] = np.exp(res.params["time_since"])
    return row

sT = series_of(pan, "ev_cvc", "pm_cvc", 1)
sC = series_of(pan, "ev_cvc", "pm_cvc", 0)
rows = []
for grp, s in [("Collaborative", sT), ("Comparison (control series)", sC)]:
    for lab, kw in [("Step only, Poisson, model-based SE (primary)", dict(full=False, cov="plain")),
                    ("Step + slope, Poisson, model-based SE", dict(full=True, cov="plain")),
                    ("Step only, quasi-Poisson", dict(full=False, cov="quasi")),
                    ("Step only, Poisson + HAC SE (sensitivity)", dict(full=False, cov="HAC"))]:
        r = its_row(lab, fit_its(s, **kw)); r["group"] = grp
        r["AIC (unscaled Poisson)"] = fit_its(s, full=kw["full"], cov="plain").aic
        rows.append(r)
its_tbl = pd.DataFrame(rows)[["group", "model", "level-change RR (exp b2)", "95% low", "95% high", "slope-change RR per month", "AIC (unscaled Poisson)"]]
display(its_tbl.round(3))
prim = its_tbl.iloc[0]
print("Truth for the collaborative group: level-change RR = 0.70, slope change = 1.00. The comparison series should show RR near 1.")
print(f"Primary ITS estimate: RR = {prim['level-change RR (exp b2)']:.3f} (95% CI {prim['95% low']:.3f} to {prim['95% high']:.3f}); "
      f"truth 0.70 is {'inside' if prim['95% low'] <= 0.7 <= prim['95% high'] else 'OUTSIDE'} this interval.")

In [ ]:
# Counterfactual picture and residual diagnostics for the primary ITS model (collaborative group)
res_T = fit_its(sT, full=False, cov="plain")
res_C = fit_its(sC, full=False, cov="plain")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8), sharey=True)
for ax, s, res, title in [(axes[0], sT, res_T, "Collaborative facilities"), (axes[1], sC, res_C, "Comparison facilities (control series)")]:
    X = its_design(s.t, full=False)
    fitted = np.exp(X.to_numpy() @ res.params.to_numpy()) * 100
    Xcf = X.copy(); Xcf["post"] = 0.0
    cf = np.exp(Xcf.to_numpy() @ res.params.to_numpy()) * 100
    ax.scatter(s.t, s.ev / s.pm * 100, color="k", s=18, label="Observed")
    ax.plot(s.t, fitted, color="crimson", lw=2, label="Fitted (with step)")
    ax.plot(s.t[s.t >= T0 - 1], cf[s.t >= T0 - 1], color="gray", ls="--", lw=2, label="Counterfactual (no step)")
    ax.axvline(T0 - 0.5, color="k", ls=":", lw=1)
    ax.set_title(title); ax.set_xlabel("Month index (0 = Jan 2023)")
axes[0].set_ylabel("CVC events per 100 CVC patient-months"); axes[0].legend(frameon=False)
plt.tight_layout(); plt.show()

resid = res_T.resid_pearson
lb = acorr_ljungbox(resid, lags=[6], return_df=True)
print(f"Residual checks (collaborative, primary model; no obvious autocorrelation or overdispersion means model-based SEs are reasonable): Durbin-Watson = {durbin_watson(resid):.2f} (2 = no autocorrelation) | "
      f"Ljung-Box(6) p = {float(lb['lb_pvalue'].iloc[0]):.3f} | Pearson chi2/df = {res_T.pearson_chi2 / res_T.df_resid:.2f}")
# seasonality check: do 12-month harmonic terms improve fit?
base = fit_its(sT, full=False, cov="plain"); seas = fit_its(sT, full=False, seasonal=True, cov="plain")
lr = 2 * (seas.llf - base.llf)
print(f"Seasonality check (sin/cos terms): LR chi2(2) = {lr:.2f}, p = {stats.chi2.sf(lr, 2):.3f}")

fig, ax = plt.subplots(figsize=(6, 2.8))
a = acf(resid, nlags=8, fft=False)
ax.bar(range(len(a)), a, color="steelblue"); ax.axhline(2 / np.sqrt(len(resid)), color="gray", ls="--"); ax.axhline(-2 / np.sqrt(len(resid)), color="gray", ls="--")
ax.set_xlabel("Lag (months)"); ax.set_ylabel("ACF"); ax.set_title("Residual autocorrelation, primary ITS model")
plt.tight_layout(); plt.show()

**How trustworthy are these ITS intervals? A simulation check.** We simulate 500 monthly series with the *same* patient-months,
the same baseline rate, a true level change of exactly 0.70, and **only Poisson noise** (the friendliest possible case), then
count how often each method's 95% interval contains 0.70. A good interval should contain it about 95% of the time.

In [ ]:
def its_coverage(s, truth=0.70, B=500, seed=11):
    r = np.random.default_rng(seed)
    base_rate = s[s.t < T0].ev.sum() / s[s.t < T0].pm.sum()
    mu = s.pm.to_numpy() * base_rate * np.where(s.t >= T0, truth, 1.0)
    methods = {"Poisson, model-based SE": "plain", "Quasi-Poisson": "quasi", "Poisson + HAC SE": "HAC"}
    hit = {k: 0 for k in methods}; est = []
    for _ in range(B):
        sim = s.assign(ev=r.poisson(mu))
        for lab, cv in methods.items():
            res = fit_its(sim, full=False, cov=cv); ci = res.conf_int().loc["post"]
            hit[lab] += bool(np.exp(ci[0]) <= truth <= np.exp(ci[1]))
            if cv == "plain":
                est.append(float(np.exp(res.params["post"])))
    return {k: v / B for k, v in hit.items()}, np.array(est)

cov_its, est_sim = its_coverage(sT)
print("Share of simulated 95% intervals that contain the true 0.70 (target 0.95):")
for k, v in cov_its.items():
    print(f"  {k:26s} {v:.3f}")
print(f"Spread of the ITS point estimate across simulations: 2.5-97.5% = {np.percentile(est_sim, 2.5):.2f} to {np.percentile(est_sim, 97.5):.2f} "
      f"(truth 0.70). A single ITS series is a noisy instrument, even with no model misspecification.")

**What ITS can and cannot tell us.** It estimates the break at the start date *assuming the pre-trend would have continued*
and *nothing else changed at that moment* ("history" threats: a new coding rule, a pandemic wave, a reporting change). The
control series is the guard against that. Where a comparison group exists, difference-in-differences uses it directly.

**Look at the counterfactual in the left panel.** A step-only model shares one slope across the whole 24 months, so a noisy upward drift in the series tilts the dashed counterfactual upward and makes the drop at the start date look larger than a simple before/after ratio would. The trend is estimated from only 24 noisy points, which is the main reason a single-series ITS is fragile (the simulation above shows the point estimate can scatter from about 0.5 to 1.0 even with a perfectly specified model) and why a comparison group helps.

## 5. Difference-in-differences (DiD)
**Estimand:** the ratio of rate ratios, `(post/pre in treated) / (post/pre in comparison)`. Under the assumptions below it is the
multiplicative effect of the collaborative on the facilities that joined (average treatment effect on the treated).

**Assumptions** (state them, then test what can be tested):
1. **Parallel trends (multiplicative):** without the collaborative, treated and comparison rates would have moved together. Testable only in the
   pre-period.
2. **No anticipation:** facilities did not change behavior before the start date.
3. **No spillover:** comparison facilities were not influenced by the collaborative.
4. **Stable composition:** the same facilities are observed throughout.

**Estimators** (all Poisson with a `log(patient-months)` offset and standard errors **clustered by facility**, because a facility's
months are not independent):
- *Naive post-only* and *naive pre-post* comparisons, shown to illustrate why each alone is weaker
- **2x2 DiD**: group x period interaction
- **Two-way fixed effects (TWFE)**: facility and month fixed effects absorb all stable facility differences and all common month shocks
- **Event study**: separate effect for every month relative to the month before the start, to test pre-trends and see the effect's shape

*TWFE caveat:* when different units adopt at *different* dates, TWFE can be biased by already-treated units acting as controls
(Goodman-Bacon). Here all treated facilities start together, so that problem does not arise. A staggered rollout would need
Callaway-Sant'Anna or Sun-Abraham estimators.

In [ ]:
def ppml(y, X, expo, groups):
    X = sm.add_constant(X.astype(float), has_constant="add")
    return sm.GLM(y, X, family=sm.families.Poisson(), offset=np.log(expo)).fit(cov_type="cluster", cov_kwds={"groups": groups})

def rr_ci(res, name):
    b, se = float(res.params[name]), float(res.bse[name])
    return np.exp(b), np.exp(b - 1.96 * se), np.exp(b + 1.96 * se)

def codes(s):
    return s.astype("category").cat.codes.to_numpy()

def twfe(p, y, expo):
    q = p[p.groupby("facility_id")[y].transform("sum") > 0].reset_index(drop=True)     # all-zero facilities carry no information in a Poisson FE model
    X = pd.concat([(q.treated * q.post).rename("treated_post"), pd.get_dummies(q.facility_id, drop_first=True),
                   pd.get_dummies(q.t, prefix="m", drop_first=True)], axis=1)
    return ppml(q[y], X, q[expo], codes(q.facility_id))

def did_suite(panel, y="ev_cvc", expo="pm_cvc"):
    p = panel[panel[expo] > 0].reset_index(drop=True)
    g = codes(p.facility_id)
    out = []
    s = (p.post == 1).to_numpy()
    out.append(("Naive: treated vs comparison, post period only",) + rr_ci(ppml(p.loc[s, y], p.loc[s, ["treated"]], p.loc[s, expo], g[s]), "treated"))
    s = (p.treated == 1).to_numpy()
    out.append(("Naive: before vs after, treated only",) + rr_ci(ppml(p.loc[s, y], p.loc[s, ["post"]], p.loc[s, expo], g[s]), "post"))
    X = p[["treated", "post"]].copy(); X["treated_post"] = p.treated * p.post
    out.append(("DiD 2x2 (group x period)",) + rr_ci(ppml(p[y], X, p[expo], g), "treated_post"))
    out.append(("DiD two-way fixed effects",) + rr_ci(twfe(p, y, expo), "treated_post"))
    return pd.DataFrame(out, columns=["estimator", "rate_ratio", "ci_low", "ci_high"])

did_cvc = did_suite(pan, "ev_cvc", "pm_cvc")
did_cvc["truth"] = 0.70
did_cvc["truth_inside_95"] = (did_cvc.truth >= did_cvc.ci_low) & (did_cvc.truth <= did_cvc.ci_high)
did_cvc["CI width (log)"] = np.log(did_cvc.ci_high / did_cvc.ci_low)
display(did_cvc.round(3))

**Reading the table honestly.** In this simulation the naive before/after estimate is both close to the truth and the most precise (narrowest interval). That is not evidence it is the better method: nothing else changes over time in the simulator (no secular trend, no seasonal or reporting shock, no regression to the mean), which is exactly the situation where a simple before/after comparison cannot go wrong. In real data it is exposed to every one of those threats, and its weakness is **bias**, not variance. The post-only comparison is wider because it must contend with permanent differences between facilities. DiD uses each facility as its own baseline *and* the comparison group as the stand-in for "what would have happened anyway", paying a little precision for protection against the common shocks that sections 6 and 7 probe. Compare the interval widths, and remember that with a single simulated dataset how close each point estimate lands to 0.70 is mostly luck.

### 5.1 Event study: pre-trends and the shape of the effect
One coefficient per month for treated facilities. Conventionally the reference is the single month before the start (December 2023),
but with sparse counts **one noisy or contaminated reference month shifts every coefficient**. We therefore plot coefficients relative to the
**average of the whole pre-period**, and also compute the single-month version for comparison. Pre-period coefficients near 1.0 support
parallel trends; post-period coefficients show whether the effect is immediate, grows, or fades. We also run a **joint Wald test
that all pre-period coefficients are zero**. A caution from the literature: a non-significant pre-trend test does not *prove*
parallel trends (it can lack power), so it is read together with the plot and subject-matter reasoning.

In [ ]:
def event_study(panel, y, expo, ref=-1, kmin=-12, kmax=11, norm="premean"):
    """Poisson event study with facility and month fixed effects.
    norm='ref'     : coefficients relative to the single month `ref`.
    norm='premean' : coefficients relative to the AVERAGE of all pre-period months (more robust when one month is noisy)."""
    q = panel[panel[expo] > 0]
    q = q[q.groupby("facility_id")[y].transform("sum") > 0].reset_index(drop=True)
    k = q.t - T0
    ks = [j for j in range(kmin, kmax + 1) if j != ref]
    ES = pd.DataFrame({f"k{j:+d}": ((q.treated == 1) & (k == j)).astype(float) for j in ks})
    X = pd.concat([ES, pd.get_dummies(q.facility_id, drop_first=True), pd.get_dummies(q.t, prefix="m", drop_first=True)], axis=1)
    res = ppml(q[y], X, q[expo], codes(q.facility_id))
    cols = list(ES.columns)
    b = res.params[cols].to_numpy(); V = res.cov_params().loc[cols, cols].to_numpy()
    kk = np.array([int(c[1:]) for c in cols])
    if norm == "premean":
        w = (kk < ref).astype(float) / ((kk < ref).sum() + 1)          # the reference month counts as one pre-period month with coefficient 0
        C = np.eye(len(cols)) - w[None, :]
        coef, var = C @ b, np.einsum("ij,jk,ik->i", C, V, C)
        coef_ref, var_ref = float(-w @ b), float(w @ V @ w)
    else:
        coef, var, coef_ref, var_ref = b, np.diag(V), 0.0, 0.0
    ks_all = np.r_[kk, ref]; c_all = np.r_[coef, coef_ref]; se_all = np.sqrt(np.r_[var, var_ref])
    tab = pd.DataFrame({"k": ks_all, "rr": np.exp(c_all), "lo": np.exp(c_all - 1.96 * se_all), "hi": np.exp(c_all + 1.96 * se_all)}).sort_values("k").reset_index(drop=True)
    def wald(sel):
        bb = b[sel]; VV = V[np.ix_(sel, sel)]
        chi = float(bb @ np.linalg.solve(VV, bb)); return chi, float(stats.chi2.sf(chi, int(sel.sum())))
    return tab, wald(kk < ref), wald(kk >= 0)

def linear_pretrend(panel, y, expo):
    """Do treated and comparison facilities diverge linearly BEFORE the program? (1 degree of freedom, facility + month fixed effects.)"""
    q = panel[(panel[expo] > 0) & (panel.t < T0)]
    q = q[q.groupby("facility_id")[y].transform("sum") > 0].reset_index(drop=True)
    X = pd.concat([(q.treated * (q.t - (T0 - 1) / 2)).rename("treated_x_time"), pd.get_dummies(q.facility_id, drop_first=True),
                   pd.get_dummies(q.t, prefix="m", drop_first=True)], axis=1)
    res = ppml(q[y], X, q[expo], codes(q.facility_id))
    rr_, lo_, hi_ = rr_ci(res, "treated_x_time")
    return rr_, lo_, hi_, float(res.pvalues["treated_x_time"])

es_cvc, w_lead_cvc, _ = event_study(pan, "ev_cvc", "pm_cvc")
es_non, w_lead_non, _ = event_study(pan, "ev_non", "pm_non")
es_cvc_ref, _, _ = event_study(pan, "ev_cvc", "pm_cvc", norm="ref")
es_non_ref, _, _ = event_study(pan, "ev_non", "pm_non", norm="ref")
print(f"Joint pre-trend test (all 11 leads = 0):  CVC events chi2 = {w_lead_cvc[0]:.1f}, p = {w_lead_cvc[1]:.3f} | non-CVC placebo chi2 = {w_lead_non[0]:.1f}, p = {w_lead_non[1]:.3f}")
lt_cvc, lt_non = linear_pretrend(pan, "ev_cvc", "pm_cvc"), linear_pretrend(pan, "ev_non", "pm_non")
print(f"Differential LINEAR pre-trend (rate ratio per month, treated vs comparison; 1 df):  CVC {lt_cvc[0]:.3f} (95% CI {lt_cvc[1]:.3f} to {lt_cvc[2]:.3f}), p = {lt_cvc[3]:.3f}"
      f"  |  non-CVC placebo {lt_non[0]:.3f} (95% CI {lt_non[1]:.3f} to {lt_non[2]:.3f}), p = {lt_non[3]:.3f}")
if w_lead_non[1] < 0.05 or w_lead_cvc[1] < 0.05:
    print("NOTE: an omnibus pre-trend test rejected. That test asks whether the treated/comparison ratio is identical in all 12 pre-period months, so a single "
          "unusual month (see the contamination table below) is enough to trigger it, even when there is no divergence over time. "
          "Read it together with the 1-df linear trend test, the plot, and the placebo DiD.")
print("(Monthly post-period coefficients are individually noisy because counts are sparse; the average post-period effect is the pooled DiD above.)")
print("Pre-period spread of the coefficients (max/min rate ratio across the 12 pre-period months):")
for lab, a_, b_ in [("  CVC, relative to pre-period mean", es_cvc, es_cvc_ref), ("  non-CVC, relative to pre-period mean", es_non, es_non_ref)]:
    pre_ = a_[a_.k < 0].rr; preR = b_[b_.k < 0].rr
    print(f"{lab}: {pre_.min():.2f} to {pre_.max():.2f}   |   relative to the single month -1: {preR.min():.2f} to {preR.max():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9), sharey=True)
for ax, es, title, truth in [(axes[0], es_cvc, "CVC events (targeted)", True), (axes[1], es_non, "Non-CVC events (placebo)", False)]:
    ax.errorbar(es.k, es.rr, yerr=[es.rr - es.lo, es.hi - es.rr], fmt="o", ms=4, capsize=2, color="steelblue")
    ax.axhline(1, color="k", lw=0.8); ax.axvline(-0.5, color="k", ls=":", lw=1)
    if truth:
        ax.hlines(0.7, 0, es.k.max(), color="crimson", ls="--", lw=1.5, label="True effect (0.70)"); ax.legend(frameon=False)
    ax.set_yscale("log"); ax.set_title(title); ax.set_xlabel("Months relative to January 2024")
axes[0].set_ylabel("Rate ratio, treated vs comparison\n(relative to pre-period average)")
plt.tight_layout(); plt.show()

**Why a reference month can mislead: contamination is not evenly spread.** Notebook 01 showed that over-reported (but plausible-looking)
events cannot be removed by rules. Because they are random in time, some months and groups get more of them. The table uses the ground
truth (simulation only) to show how many validated *non-CVC* events were silent over-reports, by month and group. A month with a cluster
of them can bend an event study that uses that single month as its reference.

In [ ]:
kv = ev_ok.merge(truth_ev, on="event_id").merge(FAC_ATTR[["facility_id", "treated"]], on="facility_id")
kv["over"] = kv.err_event.str.contains("inflated_valid")
kv["t"] = (kv.month.dt.year - 2023) * 12 + kv.month.dt.month - 1
kn = kv[~kv.access_type.isin(CVC)]
by_month = kn.groupby(["t", "treated"]).agg(events=("over", "size"), silent_over_reports=("over", "sum")).unstack("treated").fillna(0).astype(int)
by_month.columns = [f"{'collaborative' if tr else 'comparison'}: {m}" for m, tr in by_month.columns]
by_month["share over-reported (collaborative)"] = (by_month["collaborative: silent_over_reports"] / by_month["collaborative: events"]).round(2)
display(by_month.T)
worst = by_month["share over-reported (collaborative)"].idxmax()
print(f"Highest silent over-report share among collaborative non-CVC events: month index {worst} "
      f"({by_month.loc[worst, 'share over-reported (collaborative)']:.0%}); overall {kn[kn.treated == 1].over.mean():.1%} (collaborative) vs {kn[kn.treated == 0].over.mean():.1%} (comparison).")

## 6. Could this be chance or a modeling artifact?
Four independent checks:
1. **Placebo outcome:** the same DiD on non-CVC events should be about 1.
2. **Placebo date:** pretend the program started in July 2023 and use only pre-period data. Any "effect" there is bias.
3. **Permutation (randomization) test:** shuffle which facilities are called "collaborative" thousands of times. How often does a
   random label produce an effect as large as the real one? This needs no distributional assumptions.
4. **Cluster bootstrap:** resample whole facilities (within group) to get a confidence interval that does not rely on the Poisson
   model or on asymptotic standard errors.

In [ ]:
# (1) placebo outcome and all-events outcome with the full estimator suite
did_non = did_suite(pan, "ev_non", "pm_non"); did_non["truth"] = 1.00
did_all = did_suite(pan, "ev_all", "pm_all")

# design truth for the diluted all-events outcome: share of expected events that come from CVC in treated facilities after the start
post_tr = d_ok[(d_ok.month >= INTERVENTION) & d_ok.facility_id.isin(fac.loc[fac.in_collaborative, "facility_id"])]
exp_cvc = (post_tr.n_CVC_tunneled.sum() * sum(BASE_RATE["CVC_tunneled"]) + post_tr.n_CVC_nontunneled.sum() * sum(BASE_RATE["CVC_nontunneled"]))
exp_non = (post_tr.n_AVF.sum() * sum(BASE_RATE["AVF"]) + post_tr.n_AVG.sum() * sum(BASE_RATE["AVG"]) + post_tr.n_Other.sum() * sum(BASE_RATE["Other"]))
rr_all_true = (exp_non + 0.7 * exp_cvc) / (exp_non + exp_cvc)
did_all["truth"] = rr_all_true
print(f"Design truth for ALL events combined: {rr_all_true:.3f} (CVC is {exp_cvc / (exp_cvc + exp_non):.0%} of expected events in treated facilities, so the 30% CVC reduction is diluted)")
for lab, t_ in [("CVC events (target)", did_cvc), ("Non-CVC events (placebo, truth 1.00)", did_non), ("All events (diluted truth)", did_all)]:
    print("\n" + lab); display(t_[["estimator", "rate_ratio", "ci_low", "ci_high"]].round(3))

In [ ]:
# (2) placebo date: pre-period only, fake start in July 2023 (month index 6)
pre_only = pan[pan.t < T0].copy()
pre_only["post"] = (pre_only.t >= 6).astype(int)
p_ = pre_only[pre_only.pm_cvc > 0].reset_index(drop=True)
X_ = p_[["treated", "post"]].copy(); X_["treated_post"] = p_.treated * p_.post
r_ = ppml(p_.ev_cvc, X_, p_.pm_cvc, codes(p_.facility_id))
rr_, lo_, hi_ = rr_ci(r_, "treated_post")
print(f"Placebo date (fake start July 2023, pre-period only): RR = {rr_:.3f} (95% CI {lo_:.3f} to {hi_:.3f})  -> should include 1.0")

# (3) permutation test and (4) cluster bootstrap on facility-level aggregates (fast, exact 2x2 DiD ratio-of-ratios)
def fac_aggregates(panel, y="ev_cvc", expo="pm_cvc", split_t=T0, t_min=0, t_max=23):
    p = panel[(panel[expo] > 0) & (panel.t >= t_min) & (panel.t <= t_max)].copy()
    p["per"] = (p.t >= split_t).astype(int)
    a = p.groupby(["facility_id", "per"]).agg(e=(y, "sum"), pm=(expo, "sum")).unstack("per").fillna(0)
    return a.index.to_numpy(), a[("e", 0)].to_numpy(), a[("e", 1)].to_numpy(), a[("pm", 0)].to_numpy(), a[("pm", 1)].to_numpy()

def did_rr(mask, E0, E1, P0, P1):
    m = np.asarray(mask, float); c = 1.0 - m
    rate = lambda E, P, w: (E * w).sum(-1) / (P * w).sum(-1)
    return (rate(E1, P1, m) / rate(E0, P0, m)) / (rate(E1, P1, c) / rate(E0, P0, c))

ids, E0, E1, P0, P1 = fac_aggregates(pan)
treated_mask = fac.set_index("facility_id").loc[ids, "in_collaborative"].to_numpy()
rr_actual = float(did_rr(treated_mask, E0, E1, P0, P1))
print(f"2x2 ratio-of-ratios from facility aggregates: {rr_actual:.3f} (matches the Poisson 2x2 estimate above)")

N_PERM = 5000
n_tr = int(treated_mask.sum())
perm_idx = rng2.random((N_PERM, len(ids))).argsort(axis=1)[:, :n_tr]
perm_mask = np.zeros((N_PERM, len(ids)), bool)
np.put_along_axis(perm_mask, perm_idx, True, axis=1)
rr_perm = did_rr(perm_mask, E0, E1, P0, P1)
p_perm = float((np.abs(np.log(rr_perm)) >= abs(np.log(rr_actual))).mean())
print(f"Permutation test ({N_PERM:,} random relabelings): random-label RR 2.5-97.5% = {np.percentile(rr_perm, 2.5):.3f} to {np.percentile(rr_perm, 97.5):.3f}; "
      f"two-sided p = {max(p_perm, 1 / N_PERM):.4f}" + (" (below 1 in 5,000)" if p_perm == 0 else ""))

B = 2000
idx_t, idx_c = np.flatnonzero(treated_mask), np.flatnonzero(~treated_mask)
boot = np.empty(B)
for b_ in range(B):
    it_, ic_ = rng2.choice(idx_t, len(idx_t)), rng2.choice(idx_c, len(idx_c))
    ii = np.r_[it_, ic_]; m = np.r_[np.ones(len(it_)), np.zeros(len(ic_))]
    boot[b_] = did_rr(m, E0[ii], E1[ii], P0[ii], P1[ii])
print(f"Cluster bootstrap ({B:,} resamples of facilities): RR = {rr_actual:.3f}, 95% percentile CI {np.percentile(boot, 2.5):.3f} to {np.percentile(boot, 97.5):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.3))
ax[0].hist(rr_perm, bins=60, color="lightgray", edgecolor="white"); ax[0].axvline(rr_actual, color="crimson", lw=2, label=f"Actual ({rr_actual:.2f})")
ax[0].axvline(0.7, color="k", ls="--", lw=1, label="Truth (0.70)"); ax[0].set_title("Permutation distribution (labels shuffled)"); ax[0].set_xlabel("DiD rate ratio"); ax[0].legend(frameon=False)
ax[1].hist(boot, bins=50, color="steelblue", edgecolor="white"); ax[1].axvline(0.7, color="k", ls="--", lw=1)
ax[1].set_title("Cluster bootstrap distribution"); ax[1].set_xlabel("DiD rate ratio")
plt.tight_layout(); plt.show()

## 7. Selection bias: what if facilities join *because* their rates are high?
Here assignment was random, so the comparison is fair by construction. Real programs are usually voluntary or targeted at
struggling facilities, which creates **regression to the mean**: a facility chosen for an unusually bad period tends to look better
afterward even if nothing was done. To show it, we use only **2023** (no program exists yet) and invent a "program":
the 40% of facilities with the highest CVC rate in the first half of 2023 are labeled "treated" and compared with the rest over the
second half. Any apparent effect is pure selection artifact.

In [ ]:
ids23, a0, a1, q0, q1 = fac_aggregates(pan, t_min=0, t_max=11, split_t=6)
pre_rate = np.where(q0 > 0, a0 / np.maximum(q0, 1e-9), 0)
n_sel = int(round(0.4 * len(ids23)))
sel_mask = np.zeros(len(ids23), bool); sel_mask[np.argsort(-pre_rate)[:n_sel]] = True
rr_selected = float(did_rr(sel_mask, a0, a1, q0, q1))

rand_idx = rng2.random((N_PERM, len(ids23))).argsort(axis=1)[:, :n_sel]
rand_mask = np.zeros((N_PERM, len(ids23)), bool); np.put_along_axis(rand_mask, rand_idx, True, axis=1)
rr_rand = did_rr(rand_mask, a0, a1, q0, q1)
print(f"No intervention exists in 2023. 'Treated' chosen as highest early-2023 CVC rates: apparent RR = {rr_selected:.3f}")
print(f"Same comparison with RANDOM 'treated' groups: mean RR = {rr_rand.mean():.3f}, 95% range {np.percentile(rr_rand, 2.5):.3f} to {np.percentile(rr_rand, 97.5):.3f}")
print(f"Selection-on-high-rates produces a spurious {100 * (1 - rr_selected):.0f}% 'reduction' with no program at all.")

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.hist(rr_rand, bins=50, color="lightgray", edgecolor="white", label="Random assignment")
ax.axvline(rr_selected, color="crimson", lw=2, label=f"Selected on high baseline ({rr_selected:.2f})"); ax.axvline(1, color="k", lw=0.8)
ax.set_xlabel("Apparent DiD rate ratio (true effect = 1.0)"); ax.set_title("Regression to the mean in a non-randomized program"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

**Lesson for evaluating real programs.** Random assignment, or a comparison group matched on *pre-period trend* (not just pre-period
level) and tested with an event study, is what protects against this. If facilities were chosen for being bad, a plain before-after
or even a DiD against unselected comparison facilities will overstate success.

## 8. Heterogeneity, data-quality sensitivity, and public health impact

### 8.1 Did the program work equally everywhere?
The simulated effect is the same everywhere (0.70), so any apparent differences below are noise, a useful reminder of how much
uncertainty subgroup estimates carry. In real data, subgroup effects would be reported as exploratory.

In [ ]:
def did_2x2(p, y="ev_cvc", expo="pm_cvc"):
    p = p[p[expo] > 0].reset_index(drop=True)
    X = p[["treated", "post"]].copy(); X["treated_post"] = p.treated * p.post
    return rr_ci(ppml(p[y], X, p[expo], codes(p.facility_id)), "treated_post")

pan["size_group"] = pan.facility_id.map(pd.qcut(fac.set_index("facility_id").mean_census, 3, labels=["Small", "Medium", "Large"]))
rows = []
for var in ["size_group", "ownership", "urban_rural"]:
    for lvl, sub in pan.groupby(var):
        fl = sub.drop_duplicates("facility_id")
        rr3 = did_2x2(sub)
        rows.append({"subgroup": f"{var}: {lvl}", "treated facilities": int(fl.treated.sum()), "comparison facilities": int((1 - fl.treated).sum()),
                     "rate_ratio": rr3[0], "ci_low": rr3[1], "ci_high": rr3[2], "truth": 0.70})
het = pd.DataFrame(rows)
display(het.round(3))

### 8.2 What does skipping data validation cost?
The same DiD estimators on three versions of the data: **clean** (truth, only exists in simulation), **validated** (notebook 01's
rules applied), and **as submitted** (no validation: typos, duplicates, bad dates, mis-assigned access types, late data).

In [ ]:
rows = []
for lab, panel in [("Clean (simulation truth)", pan_clean), ("Validated (notebook 01 rules)", pan), ("As submitted (no validation)", pan_raw)]:
    s = did_suite(panel, "ev_cvc", "pm_cvc").set_index("estimator")
    for est in ["DiD 2x2 (group x period)", "DiD two-way fixed effects"]:
        r = s.loc[est]
        rows.append({"data": lab, "estimator": est, "rate_ratio": r.rate_ratio, "ci_low": r.ci_low, "ci_high": r.ci_high,
                     "CI width (log)": np.log(r.ci_high / r.ci_low)})
dq = pd.DataFrame(rows)
display(dq.round(3))
print("How to read this table: compare interval WIDTH, which grows as data quality worsens (more noise, less precision). Do not compare how close "
      "each point estimate lands to 0.70: with a single simulated dataset, that gap is dominated by sampling luck, and a worse dataset can land "
      "near the truth by chance. Judging bias would require repeating the whole simulation many times. The planted errors were random with respect "
      "to treatment, so they add noise rather than systematic bias; if errors were more common in one group (for example, the facilities that "
      "struggle with reporting also struggle with infections), bias would appear.")

### 8.3 Events averted
Translate the rate ratio into a number people can act on: how many CVC-related events did the collaborative prevent in 2024?
`averted = observed / RR - observed`, with the interval from the cluster bootstrap above.

In [ ]:
obs_post_T = int(pan[(pan.treated == 1) & (pan.post == 1)].ev_cvc.sum())
averted_pt = obs_post_T * (1 / rr_actual - 1)
averted_ci = np.percentile(obs_post_T * (1 / boot - 1), [2.5, 97.5])
averted_design = obs_post_T * (1 / 0.7 - 1)
print(f"Observed CVC events in collaborative facilities, 2024: {obs_post_T}")
print(f"Estimated events averted: {averted_pt:.0f} (95% bootstrap interval {averted_ci[0]:.0f} to {averted_ci[1]:.0f}) | design-truth value: {averted_design:.0f}")
print(f"\nPlain-language summary: After the collaborative started, CVC-related events in participating facilities fell by about "
      f"{100 * (1 - rr_actual):.0f}% relative to comparison facilities (rate ratio {rr_actual:.2f}, 95% interval "
      f"{np.percentile(boot, 2.5):.2f} to {np.percentile(boot, 97.5):.2f}). That corresponds to roughly {averted_pt:.0f} fewer CVC-related events in 2024. "
      "Events that do not involve a CVC did not change, which supports attributing the drop to the CVC-focused program rather than to a "
      "general reporting change. Because the reduction applies to CVC events only, the effect on all events combined is smaller.")

In [ ]:
# Summary forest plot: every estimator against the truth
est_rows = []
for _, r in did_cvc.iterrows():
    est_rows.append((r.estimator, r.rate_ratio, r.ci_low, r.ci_high))
r_its = its_row("x", res_T); est_rows.append(("ITS step-only (treated group)", r_its["level-change RR (exp b2)"], r_its["95% low"], r_its["95% high"]))
est_rows.append(("Cluster bootstrap (2x2)", rr_actual, float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))))
fe = pd.DataFrame(est_rows, columns=["estimator", "rr", "lo", "hi"])
fig, ax = plt.subplots(figsize=(8, 3.6))
yy = np.arange(len(fe))[::-1]
ax.errorbar(fe.rr, yy, xerr=[fe.rr - fe.lo, fe.hi - fe.rr], fmt="o", capsize=3, color="steelblue")
ax.axvline(0.7, color="crimson", ls="--", label="True effect (0.70)"); ax.axvline(1, color="k", lw=0.8)
ax.set_yticks(yy); ax.set_yticklabels(fe.estimator, fontsize=8); ax.set_xlabel("Rate ratio for CVC events (95% CI)")
ax.set_title("All estimators vs the known truth"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
its_tbl.to_csv(f"{OUT}/its_results.csv", index=False)
did_cvc.to_csv(f"{OUT}/did_cvc.csv", index=False)
es_cvc.to_csv(f"{OUT}/event_study_cvc.csv", index=False)
with pd.ExcelWriter(f"{OUT}/collaborative_evaluation.xlsx", engine="openpyxl") as xw:
    its_tbl.to_excel(xw, sheet_name="ITS", index=False)
    did_cvc.to_excel(xw, sheet_name="DiD CVC (target)", index=False)
    did_non.to_excel(xw, sheet_name="DiD non-CVC (placebo)", index=False)
    did_all.to_excel(xw, sheet_name="DiD all events", index=False)
    es_cvc.to_excel(xw, sheet_name="Event study CVC", index=False)
    es_non.to_excel(xw, sheet_name="Event study non-CVC", index=False)
    pd.DataFrame([{"outcome": "CVC", "omnibus_pretrend_p": w_lead_cvc[1], "linear_pretrend_RR_per_month": lt_cvc[0], "linear_pretrend_p": lt_cvc[3]},
                  {"outcome": "non-CVC placebo", "omnibus_pretrend_p": w_lead_non[1], "linear_pretrend_RR_per_month": lt_non[0], "linear_pretrend_p": lt_non[3]}]).to_excel(xw, sheet_name="Pre-trend tests", index=False)
    es_cvc_ref.to_excel(xw, sheet_name="ES CVC (ref month -1)", index=False)
    es_non_ref.to_excel(xw, sheet_name="ES non-CVC (ref month -1)", index=False)
    pd.DataFrame([{"method": k, "coverage_of_truth": v} for k, v in cov_its.items()]).to_excel(xw, sheet_name="ITS coverage simulation", index=False)
    het.to_excel(xw, sheet_name="Heterogeneity", index=False)
    dq.to_excel(xw, sheet_name="Data quality sensitivity", index=False)
    pd.DataFrame({"check": ["Permutation p (two-sided)", "Bootstrap CI low", "Bootstrap CI high", "Selection-bias apparent RR (no program exists)",
                            "Events averted (point)", "Events averted low", "Events averted high"],
                  "value": [max(p_perm, 1 / N_PERM), np.percentile(boot, 2.5), np.percentile(boot, 97.5), rr_selected,
                            averted_pt, averted_ci[0], averted_ci[1]]}).to_excel(xw, sheet_name="Robustness", index=False)
    for ws in xw.book.worksheets:
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(50, max(11, max(len(str(c.value or "")) for c in col[:40]) + 2))
        ws.freeze_panes = "A2"
print(sorted(os.listdir(OUT)))

## 9. So what for public health?
- **Use a comparison group whenever you can.** ITS is useful when no comparison exists, but a single series is a noisy instrument
  (see the simulation: the estimate scatters widely even with perfect specification), it depends on nothing else changing at the start date,
  and HAC standard errors can be too narrow with only 24 points. DiD plus an event study makes the assumptions visible and testable.
- **Do not trust one reference month.** With sparse counts, one noisy or contaminated month can shift an entire event study. Normalize to
  the pre-period average, and look at where data-quality problems landed.
- **Placebo outcomes are cheap and persuasive, but read them with judgment.** The placebo DiD for non-CVC events showed no statistically
  detectable change, which supports attributing the CVC drop to the CVC-focused program. The placebo's omnibus pre-trend test, however, rejected
  in this run because of one contaminated month, a reminder that diagnostic tests can fail for mundane data reasons and should be
  investigated (here, traced to silent over-reporting), not just reported as pass/fail.
- **Report effects on the scale people use.** A rate ratio of 0.7 on CVC events becomes a smaller change in total events and a
  concrete number of events averted, with an honest interval.
- **Regression to the mean is the main threat in real programs.** The selection demonstration shows how a program aimed at
  high-rate facilities can look successful with no effect at all. Design the evaluation (randomization, staggered rollout,
  matched comparison on pre-trends) before the program starts, not after.
- **Validation is part of the evaluation.** Poor data quality adds noise (and bias, if errors correlate with treatment). Notebook 01's
  pipeline is what makes this analysis defensible.

## Limitations
- The effect is homogeneous, immediate, and instantaneous in the simulator. Real effects ramp up, fade, and vary by site.
- Random assignment makes parallel trends true by construction; real programs rarely have that luxury.
- All treated facilities start together, so TWFE is valid here. Staggered adoption needs modern DiD estimators.
- No spillover or anticipation is simulated. Real facilities in the same network or chain may share practices.
- Standard errors assume the facility is the independent unit. Real data may also have network- or chain-level clustering.
- Only 24 months and one intervention date. Longer pre-periods make pre-trend tests more informative.

## Next notebooks
`03b` early-warning charts (CUSUM/EWMA), cluster detection, and forecasting (SARIMAX, Prophet, LSTM) | `04` outreach prioritization with a tabular
neural network and SHAP | `05` inquiry triage, facility reports, training materials.